In [ ]:
!rm -rf /kaggle/working/visual_search

In [ ]:
!git clone https://github.com/Nihit3003/Visual-Product-Search-Engine /kaggle/working/visual_search

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import torchvision
import torch

print(np.__version__)

print("Torchvision OK")

In [ ]:
!pip install -q hnswlib

In [ ]:
import subprocess
import sys
import torch

packages = [

    "hnswlib",

    "open-clip-torch==2.24.0",

    "transformers==4.41.2",

    "accelerate==0.28.0",

    "sentencepiece",

    "ultralytics==8.3.0",

    "streamlit==1.33.0",

    "pyngrok",

    "timm",

    "einops",

]

for pkg in packages:

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        check=True
    )

print("All packages installed")

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():

    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os
import sys

PROJECT_ROOT = "/kaggle/working/visual_search"

sys.path.insert(0, PROJECT_ROOT)

# =========================================================
# DATASET
# =========================================================

DATASET_ROOT = (
    "/kaggle/input/datasets/"
    "abhinavkishan123/"
    "deepfashion-inshop-dataset/"
    "Dataset"
)

# =========================================================
# OUTPUTS
# =========================================================

OUTPUT_DIR = "/kaggle/working"

CKPT_DIR = f"{OUTPUT_DIR}/checkpoints"

INDEX_DIR = f"{OUTPUT_DIR}/index"

RESULTS_DIR = f"{OUTPUT_DIR}/results"

CAPTION_DIR = f"{OUTPUT_DIR}/captions"

# =========================================================
# CREATE DIRS
# =========================================================

for d in [

    CKPT_DIR,

    INDEX_DIR,

    RESULTS_DIR,

    CAPTION_DIR,

]:
    os.makedirs(d, exist_ok=True)

# =========================================================
# VERIFY DATASET
# =========================================================

required_files = [

    "list_eval_partition.txt",

    "list_bbox_inshop.txt",

    "img",

]

print("Checking dataset files...\n")

for f in required_files:

    path = os.path.join(
        DATASET_ROOT,
        f
    )

    if os.path.exists(path):

        print(f"✓ {path}")

    else:

        print(f"✗ {path}")

print("\nDirectories ready.")

In [ ]:
content = """
# Visual Product Search Engine — Source Package

from .dataset import (
    build_dataloaders,
    get_clip_transform,
    parse_eval_partition,
    parse_bboxes,
    infer_category,
    DeepFashionDataset,
    bbox_crop,
)

from .model import (
    VisualSearchModel,
    SupConLoss,
)

from .blip_module import (
    FashionCaptioner,
    ITMReranker,
)

from .localizer import (
    YOLOLocalizer,
)

from .index import (
    HNSWIndex,
)

from .metrics import (
    evaluate,
    MetricResults,
    evaluate_multi_seed,
)
"""

with open(
    "/kaggle/working/visual_search/src/__init__.py",
    "w"
) as f:

    f.write(content)

print("Fixed __init__.py")

In [ ]:
from src.dataset import parse_eval_partition

splits, img_to_item = parse_eval_partition(
    f'{DATASET_ROOT}/list_eval_partition.txt'
)

for k, v in splits.items():
    print(f'{k:8s}: {len(v):,} images')

In [ ]:
import os

SEEDS = [2, 536, 576, 584]

for seed in SEEDS:

    print(f"\n===== Seed {seed} =====")

    cmd = f"""
python /kaggle/working/visual_search/scripts/train_clip.py \
    --dataset_root "{DATASET_ROOT}" \
    --output_dir "{CKPT_DIR}/seed_{seed}" \
    --epochs 15 \
    --batch_size 48 \
    --lr 2e-5 \
    --weight_decay 1e-2 \
    --unfreeze_last_n 6 \
    --embed_dim 768 \
    --triplet_margin 0.25 \
    --triplet_weight 0.35 \
    --temperature 0.07 \
    --seed {seed} \
    --eval_every 1
"""

    print(cmd)

    exit_code = os.system(cmd)

    if exit_code != 0:

        raise RuntimeError(
            f"Training failed for seed {seed}"
        )

In [ ]:
import json
from pathlib import Path

SEEDS = [2, 536, 576, 584]

best_seed = None
best_r10 = -1

for seed in SEEDS:

    hist_path = (
        Path(CKPT_DIR)
        / f"seed_{seed}"
        / "train_history.json"
    )

    if not hist_path.exists():

        print(f"Missing history for seed {seed}")

        continue

    with open(hist_path) as f:

        history = json.load(f)

    if len(history) == 0:
        continue

    seed_best = max(
        h.get("recall@10", 0.0)
        for h in history
    )

    print(
        f"Seed {seed} "
        f"best Recall@10 = "
        f"{seed_best:.4f}"
    )

    if seed_best > best_r10:

        best_r10 = seed_best

        best_seed = seed

if best_seed is None:

    raise RuntimeError(
        "No valid checkpoints found."
    )

best_ckpt = {

    "best_seed":
        best_seed,

    "best_recall@10":
        best_r10,

    "path":
        f"{CKPT_DIR}/seed_{best_seed}/clip_finetuned_best.pt"
}

with open(
    f"{RESULTS_DIR}/best_checkpoint.json",
    "w"
) as f:

    json.dump(
        best_ckpt,
        f,
        indent=2
    )

print("\nBest checkpoint selected:")

print(json.dumps(
    best_ckpt,
    indent=2
))

In [ ]:
import json

with open(f"{RESULTS_DIR}/best_checkpoint.json") as f:

    best_ckpt = json.load(f)

CKPT_PATH = best_ckpt["path"]

BEST_SEED = best_ckpt["best_seed"]

BEST_R10 = best_ckpt["best_recall@10"]

print(f"Best seed      : {BEST_SEED}")

print(f"Best Recall@10 : {BEST_R10:.4f}")

print(f"Checkpoint     : {CKPT_PATH}")

In [ ]:
!python /kaggle/working/visual_search/scripts/build_index.py \
    --dataset_root "{DATASET_ROOT}" \
    --index_dir "{INDEX_DIR}" \
    --condition A \
    --alpha 1.0 \
    --batch_size 48 \
    --embed_dim 768 \
    --use_gt_bbox

In [ ]:
!python /kaggle/working/visual_search/scripts/build_index.py \
    --dataset_root "{DATASET_ROOT}" \
    --captions_json "{CAPTION_DIR}/captions.json" \
    --index_dir "{INDEX_DIR}" \
    --condition B \
    --alpha 0.7 \
    --batch_size 48 \
    --embed_dim 768 \
    --use_gt_bbox

In [ ]:
!python /kaggle/working/visual_search/scripts/build_index.py \
    --dataset_root "{DATASET_ROOT}" \
    --captions_json "{CAPTION_DIR}/captions.json" \
    --ckpt_path "{CKPT_PATH}" \
    --index_dir "{INDEX_DIR}" \
    --condition C \
    --alpha 0.7 \
    --batch_size 48 \
    --embed_dim 768 \
    --use_gt_bbox

In [ ]:
!python /kaggle/working/visual_search/scripts/evaluate.py \
    --dataset_root "{DATASET_ROOT}" \
    --index_base "{INDEX_DIR}" \
    --ckpt_path "{CKPT_PATH}" \
    --output_dir "{RESULTS_DIR}" \
    --embed_dim 768 \
    --batch_size 48 \
    --alpha_B 0.7 \
    --alpha_C 0.7 \
    --num_queries 1000 \
    --seeds 2 536 576 584

In [ ]:
import json
import pandas as pd

results_path = (
    f"{RESULTS_DIR}/ablation_results.json"
)

with open(results_path) as f:

    res = json.load(f)

metric_order = [

    "Recall@5",
    "Recall@10",
    "Recall@15",

    "NDCG@5",
    "NDCG@10",
    "NDCG@15",

    "mAP@5",
    "mAP@10",
    "mAP@15",
]

rows = []

for cond, metrics in res.items():

    row = {

        "Condition":
            cond.replace(
                "condition_",
                ""
            )
    }

    for metric in metric_order:

        vals = metrics.get(metric)

        if not isinstance(vals, dict):

            row[metric] = "-"

            continue

        mean = vals.get("mean")

        std = vals.get("std")

        if mean is None or std is None:

            row[metric] = "-"

            continue

        row[metric] = (
            f"{mean:.4f} ± "
            f"{std:.4f}"
        )

    rows.append(row)

df = pd.DataFrame(rows)

if not df.empty:

    df = df.set_index(
        "Condition"
    )

    df = df[metric_order]

    print("\nFinal Ablation Results\n")

    print(df.to_string())

    display(df)

else:

    print(
        "No evaluation results found."
    )

In [ ]:
import os

print("\n===== IMPORTANT OUTPUTS =====\n")

important_paths = [

    CKPT_PATH,

    f"{INDEX_DIR}/condition_C_alpha0.7/hnsw.bin",

    f"{INDEX_DIR}/condition_C_alpha0.7/metadata.json",

    f"{INDEX_DIR}/condition_C_alpha0.7/index_config.json",

    f"{RESULTS_DIR}/ablation_results.json",

    f"{RESULTS_DIR}/best_checkpoint.json",

    f"{CAPTION_DIR}/captions.json",
]

for p in important_paths:

    exists = os.path.exists(p)

    status = "✓" if exists else "✗"

    print(f"{status} {p}")